#  XGBoost  Training(2013—2019) - Respira SP

Este notebook dá continuidade à preparação dos dados, utilizando os datasets supervisionados gerados anteriormente para treinar, regularizar e avaliar modelos XGBoost na previsão de PM2.5 24 horas à frente.

Foram comparadas três estratégias de pré-processamento para tratamento de gaps, além de diferentes configurações do XGBoost: modelo inicial, modelo regularizado, modelo com novas features temporais e modelo regularizado com novas features temporais.

## O que foi feito

- Carregamento dos datasets supervisionados referentes às três estratégias de pré-processamento;
- Remoção da coluna `MP2.5` original para evitar seu uso direto como feature no modelo;
- Definição das features utilizadas no treinamento e do target `target_24h`;
- Realização do `split temporal` dos dados:
    - Treino: dados ***anteriores a 2018***;
    - Validação: ***dados de 2018***;
    - Teste: ***dados de 2019***;
- Criação de uma pipeline com SimpleImputer, garantindo que o ajuste do imputer seja feito apenas nos dados de treino;
- Transformação dos dados de validação e teste utilizando apenas o objeto ajustado no treino, evitando data leakage;
- Treinamento de um baseline simples baseado em `PM2.5_lag_24h`;
- Treinamento inicial do modelo ***XGBoost*** para cada estratégia de pré-processamento;
- Avaliação dos modelos com `MAE, RMSE e R²`;
- Comparação entre `baseline` e `XGBoost` no conjunto de validação;
- Avaliação final do `XGBoost` no conjunto de teste;
- Análise de `feature importance`;
- Análise de `desempenho por split`;
- Construção de `learning curves` para investigar possíveis sinais de overfitting, underfitting ou data leakage.
- Realização de `GridSearchCV` com `TimeSeriesSplit` para buscar configurações mais regularizadas;
- Treinamento de modelos `XGBoost` regularizados usando os melhores hiperparâmetros encontrados;
- Comparação entre modelos iniciais e modelos regularizados;
- Criação de novas features temporais:
    - `hora_sin / hora_cos`;
    - `mes_sin / mes_cos`;
    - `dia_semana_sin / dia_semana_cos`;
    - `PM2.5_diff_1h`;
    - `PM2.5_diff_3h`;
    - `PM2.5_diff_6h`;
    - `PM2.5_diff_24h`;
- Treinamento de XGBoost inicial com as novas features temporais;
- Treinamento de XGBoost regularizado com as novas features temporais;
- Comparação final entre:
    - `Modelos iniciais`;
    - `Modelos regularizados`;
    - `Modelos iniciais com features temporais`;
    - `Modelos regularizados com features temporais`.


## Observação

Nesta etapa, o foco foi comparar diferentes versões do `XGBoost` para previsão de `MP2.5` 24 horas à frente. A regularização reduziu sinais de overfitting nas learning curves, mas não trouxe ganho consistente no conjunto de teste. As novas features temporais também não apresentaram melhora expressiva e consistente nas métricas finais.

De forma geral, os modelos apresentaram desempenhos próximos entre as três estratégias de pré-processamento. A seleção final deve considerar principalmente o desempenho em validação, a estabilidade entre validação e teste e a simplicidade da configuração.

A base utilizada corresponde ao período histórico de ***2013 a 2019*** e mantém as seguintes features meteorológicas:

- `temperature_2m`
- `relative_humidity_2m`
- `precipitation`
- `wind_speed_10m`

A ideia é comparar esta abordagem com o pré-processamento anterior do grupo, avaliando posteriormente qual estratégia gera melhor desempenho nos modelos.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor
import matplotlib.pyplot as plt
from sklearn.model_selection import learning_curve, TimeSeriesSplit, GridSearchCV

In [2]:
# Importando as bases de dados 2016-2019
# Estrategia 1 sem imterpolação temporal / sazonal
df_estrategia01 = pd.read_csv("../data/data_sem_interpolacao/dados_pm25_umi_temp_2016_2019_nan.csv")
df_estrategia01.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/data_sem_interpolacao/dados_pm25_umi_temp_2016_2019_nan.csv'